In [ ]:
# Full Autoencoder training + extended visual diagnostics on MNIST

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# --------------------
# Configuration
# --------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 128
epochs = 10
latent_dim = 32
lr = 1e-3

# --------------------
# Data
# --------------------
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# --------------------
# Model
# --------------------
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),   # 14x14
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), # 7x7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim)
        )

        self.decoder_fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), # 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1),  # 28x28
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x = self.decoder_fc(z)
        x = x.view(-1, 64, 7, 7)
        return self.decoder(x)

model = Autoencoder(latent_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# --------------------
# Training
# --------------------
losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for x, _ in train_loader:
        x = x.to(device)
        x_hat = model(x)
        loss = criterion(x_hat, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

# --------------------
# Loss plot
# --------------------
plt.figure()
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss – Autoencoder")
plt.show()

# ============================================================
# VISUAL DIAGNOSTICS
# ============================================================

# --------------------
# 1. Reconstruction + Error map
# --------------------
model.eval()
with torch.no_grad():
    images, _ = next(iter(test_loader))
    images = images.to(device)
    recon = model(images)

images = images.cpu()
recon = recon.cpu()
error = torch.abs(images - recon)

n = 6
plt.figure(figsize=(18, 7))

for i in range(n):
    plt.subplot(3, n, i + 1)
    plt.imshow(images[i][0], cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(3, n, i + 1 + n)
    plt.imshow(recon[i][0], cmap="gray")
    plt.title("Reconstruction")
    plt.axis("off")

    plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(error[i][0], cmap="inferno")
    plt.title("|Error|")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.axis("off")

plt.suptitle("Original vs Reconstruction vs Error")
plt.tight_layout()
plt.show()

# --------------------
# 2. Latent space visualization (PCA)
# --------------------
latents = []
labels = []

with torch.no_grad():
    for x, y in test_loader:
        z = model.encoder(x.to(device))
        latents.append(z.cpu())
        labels.append(y)

latents = torch.cat(latents).numpy()
labels = torch.cat(labels).numpy()

z_2d = PCA(n_components=2).fit_transform(latents)

plt.figure(figsize=(6, 6))
scatter = plt.scatter(
    z_2d[:, 0],
    z_2d[:, 1],
    c=labels,
    cmap="tab10",
    s=5,
    alpha=0.7
)
plt.colorbar(scatter)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Latent Space (PCA)")
plt.show()

# --------------------
# 3. Latent space interpolation
# --------------------
model.eval()
x1, _ = test_dataset[0]
x2, _ = test_dataset[1]

with torch.no_grad():
    z1 = model.encoder(x1.unsqueeze(0).to(device))
    z2 = model.encoder(x2.unsqueeze(0).to(device))

    alphas = torch.linspace(0, 1, 8)
    recon_interp = []

    for a in alphas:
        z = (1 - a) * z1 + a * z2
        x_hat = model.decoder(model.decoder_fc(z))
        recon_interp.append(x_hat.cpu())

plt.figure(figsize=(12, 2))
for i, img in enumerate(recon_interp):
    plt.subplot(1, 8, i + 1)
    plt.imshow(img[0][0], cmap="gray")
    plt.axis("off")

plt.suptitle("Latent Space Interpolation")
plt.show()

# --------------------
# Reconstruction sensitivity heatmap (saliency)
# --------------------
model.eval()

x, _ = test_dataset[0]
x = x.unsqueeze(0).to(device)
x.requires_grad = True

x_hat = model(x)
loss = ((x_hat - x) ** 2).mean()
loss.backward()

saliency = x.grad.abs().cpu()[0, 0]

plt.figure(figsize=(4, 4))
plt.imshow(saliency, cmap="hot")
plt.colorbar()
plt.title("Reconstruction Sensitivity Heatmap")
plt.axis("off")
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

LATENT_DIM = 100
BATCH_SIZE = 128
EPOCHS = 10
LR = 2e-4

# ---- Dane ----
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# ---- Modele ----
class Generator(nn.Module):
    def __init__(self, z_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, device=device)

# ---- Trening ----
for epoch in range(1, EPOCHS + 1):
    for real, _ in dl:
        real = real.to(device)
        bs = real.size(0)

        # --- uczymy Dyskryminator ---
        z = torch.randn(bs, LATENT_DIM, device=device)
        fake = G(z).detach()

        pred_real = D(real)
        pred_fake = D(fake)

        y_real = torch.ones(bs, 1, device=device)
        y_fake = torch.zeros(bs, 1, device=device)

        loss_D = criterion(pred_real, y_real) + criterion(pred_fake, y_fake)

        opt_D.zero_grad(set_to_none=True)
        loss_D.backward()
        opt_D.step()

        # --- uczymy Generator ---
        z = torch.randn(bs, LATENT_DIM, device=device)
        fake = G(z)
        pred = D(fake)

        loss_G = criterion(pred, y_real)  # generator chce "oszukać" Dyskryminator

        opt_G.zero_grad(set_to_none=True)
        loss_G.backward()
        opt_G.step()

    with torch.no_grad():
        samples = G(fixed_noise).cpu()
    grid = utils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    utils.save_image(grid, f"gan_samples_epoch_{epoch:02d}.png")
    print(f"Epoch {epoch}/{EPOCHS} | loss_D={loss_D.item():.4f} | loss_G={loss_G.item():.4f}")


Epoch 1/10 | loss_D=0.9455 | loss_G=1.1338
Epoch 2/10 | loss_D=0.6887 | loss_G=2.1831
Epoch 3/10 | loss_D=0.9150 | loss_G=1.7007
Epoch 4/10 | loss_D=1.2127 | loss_G=2.6536
Epoch 5/10 | loss_D=1.2516 | loss_G=3.6577
Epoch 6/10 | loss_D=0.9416 | loss_G=2.9222
Epoch 7/10 | loss_D=0.7879 | loss_G=1.5769
Epoch 8/10 | loss_D=1.8023 | loss_G=0.6054
Epoch 9/10 | loss_D=1.0281 | loss_G=0.9555
Epoch 10/10 | loss_D=0.7912 | loss_G=1.3205


In [ ]:
# pip install torch torchvision matplotlib scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ----------------------------
# Config
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

LATENT_DIM = 128
BATCH_SIZE = 128
EPOCHS = 30
LR = 2e-4
THRESH = 0.5

OUT_DIR = "./gan_cifar10_out"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# Data: CIFAR-10 (3x32x32)
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))  # -> [-1,1]
])

train_ds = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# ----------------------------
# Models (DCGAN-style)
# ----------------------------
class Generator(nn.Module):
    def __init__(self, z_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()  # RGB in [-1,1]
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

# ----------------------------
# Train GAN
# ----------------------------
for epoch in range(1, EPOCHS + 1):
    G.train()
    D.train()

    for real_imgs, _ in train_dl:
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        # ---- Train D ----
        z = torch.randn(bs, LATENT_DIM, 1, 1, device=device)
        fake_imgs = G(z).detach()

        y_real = torch.ones(bs, 1, device=device)
        y_fake = torch.zeros(bs, 1, device=device)

        loss_D = (
            criterion(D(real_imgs), y_real) +
            criterion(D(fake_imgs), y_fake)
        )

        opt_D.zero_grad(set_to_none=True)
        loss_D.backward()
        opt_D.step()

        # ---- Train G ----
        z = torch.randn(bs, LATENT_DIM, 1, 1, device=device)
        fake_imgs = G(z)
        loss_G = criterion(D(fake_imgs), y_real)

        opt_G.zero_grad(set_to_none=True)
        loss_G.backward()
        opt_G.step()

    # Save samples
    G.eval()
    with torch.no_grad():
        samples = G(fixed_noise).cpu()
    grid = utils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    utils.save_image(grid, f"{OUT_DIR}/samples_epoch_{epoch:03d}.png")

    print(f"Epoch {epoch}/{EPOCHS} | loss_D={loss_D.item():.4f} | loss_G={loss_G.item():.4f}")

# ----------------------------
# Evaluation: D vs TRUE labels
# ----------------------------
G.eval()
D.eval()

N_REAL = 2000
N_FAKE = 2000

# Real images
real_imgs = []
it = iter(train_dl)
while len(real_imgs) * BATCH_SIZE < N_REAL:
    batch, _ = next(it)
    real_imgs.append(batch)
real_imgs = torch.cat(real_imgs, dim=0)[:N_REAL].to(device)

# Fake images
z = torch.randn(N_FAKE, LATENT_DIM, 1, 1, device=device)
with torch.no_grad():
    fake_imgs = G(z)

# Stack + labels
x_eval = torch.cat([real_imgs, fake_imgs], dim=0)
y_true = torch.cat([
    torch.ones(N_REAL),
    torch.zeros(N_FAKE)
], dim=0).numpy()

with torch.no_grad():
    scores = D(x_eval).squeeze(1).cpu().numpy()

y_pred = (scores >= THRESH).astype(int)

# Metrics
cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
print("\nConfusion matrix (rows=true [real,fake], cols=pred [real,fake]):\n", cm)
print("\nClassification report (1=real, 0=fake):\n")
print(classification_report(y_true, y_pred, digits=4))

# ----------------------------
# Visualizations
# ----------------------------

# Confusion matrix
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xticks([0, 1], ["pred real", "pred fake"])
plt.yticks([0, 1], ["true real", "true fake"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=160)
plt.close()

# Score distributions
plt.figure(figsize=(6, 4))
plt.hist(scores[:N_REAL], bins=40, alpha=0.6, label="real")
plt.hist(scores[N_REAL:], bins=40, alpha=0.6, label="fake")
plt.axvline(THRESH, linestyle="--", linewidth=2)
plt.legend()
plt.title("Discriminator score distributions")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/score_distributions.png", dpi=160)
plt.close()

# Most fooled fake images
k = 16
top_idx = np.argsort(scores[N_REAL:])[-k:]
grid = utils.make_grid(fake_imgs[top_idx].cpu(), nrow=4,
                       normalize=True, value_range=(-1, 1))
utils.save_image(grid, f"{OUT_DIR}/top_fooled_fakes.png")

print(f"\nWyniki zapisane w: {OUT_DIR}")


100%|██████████| 170M/170M [00:04<00:00, 42.5MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1/30 | loss_D=0.6240 | loss_G=2.5700
Epoch 2/30 | loss_D=0.5749 | loss_G=1.8948


**Cel zadania 1**
Celem zadania jest zaprojektowanie, implementacja oraz analiza działania generatywnej sieci adwersarialnej (GAN), zdolnej do generowania nowych danych podobnych do danych uczących.

**Opis zadania**
Zaimplementuj model GAN składający się z dwóch sieci neuronowych:

* **generatora (G)**, którego zadaniem jest generowanie syntetycznych próbek danych,
* **dyskryminatora (D)**, którego zadaniem jest rozróżnianie danych rzeczywistych od danych wygenerowanych.

Model powinien zostać wytrenowany w schemacie uczenia adwersarialnego, w którym generator i dyskryminator konkurują ze sobą.

**Zakres**

1. Wybierz zbiór dwa zbiory danych (np. MNIST, Fashion-MNIST, CIFAR-10 lub prosty zbiór syntetyczny).
2. Zdefiniuj architekturę generatora i dyskryminatora (liczba warstw, funkcje aktywacji).
3. Zaimplementuj funkcję straty dla obu sieci oraz procedurę treningu.
4. Przeprowadź proces uczenia modelu.
5. Zwizualizuj wygenerowane próbki danych na różnych etapach treningu.
6. Przeanalizuj stabilność treningu oraz jakość generowanych danych.


In [ ]:
# Zadanie 1: GAN – trening (MNIST + Fashion-MNIST)
# %pip install torch torchvision matplotlib

%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab03.gan import GANConfig, Discriminator, Generator, display_results_in_notebook, train_gan

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

cfg = GANConfig(
    latent_dim=100,
    batch_size=128,
    epochs=15,
    lr=2e-4,
    snapshot_epochs=(1, 5, 10, 15),
    num_workers=0,
)

OUT = Path("./outputs/lab03_zadanie1_gan")
histories = {}

for name in ("mnist", "fashion_mnist"):
    title = "MNIST" if name == "mnist" else "Fashion-MNIST"
    print(f"\n{'='*50}\nTrening GAN: {title}\n{'='*50}")
    histories[title] = train_gan(
        dataset_name=name,
        out_dir=OUT / name,
        device=device,
        cfg=cfg,
        seed=42 if name == "mnist" else 123,
    )

G = Generator(cfg.latent_dim)
D = Discriminator()
print("\n--- Generator (G) ---")
print(G)
print(f"Parametry G: {sum(p.numel() for p in G.parameters()):,}")
print("\n--- Dyskryminator (D) ---")
print(D)
print(f"Parametry D: {sum(p.numel() for p in D.parameters()):,}")
print(f"\nPliki zapisane w: {OUT.resolve()}")

# Wyświetl wykresy i obrazki bezpośrednio w notebooku
display_results_in_notebook(OUT, histories=histories, snapshot_epochs=cfg.snapshot_epochs)


In [ ]:
# Zadanie 1: podgląd wyników w notebooku (bez ponownego treningu)
%matplotlib inline

from pathlib import Path
from lab03.gan import GANConfig, display_results_in_notebook

OUT = Path("./outputs/lab03_zadanie1_gan")
cfg = GANConfig()

if not (OUT / "mnist").exists():
    raise FileNotFoundError(
        f"Brak wyników w {OUT}. Najpierw uruchom komórkę treningu powyżej."
    )

display_results_in_notebook(OUT, histories=None, snapshot_epochs=cfg.snapshot_epochs)
print("Gotowe – wykresy i siatki powyżej.")


### Analiza (Zadanie 1)

**Architektura:** DCGAN – G: ConvTranspose2d + BatchNorm + ReLU + Tanh; D: Conv2d + BatchNorm + LeakyReLU(0.2) + Sigmoid.

**Funkcje straty:** binarna entropia krzyżowa (BCE) – D uczy się klasyfikować real/fake; G maksymalizuje P(fake=real).

**Stabilność:** lekkie label smoothing (real=0.9, fake=0.1) ogranicza przeuczenie D. Obserwuj wykresy loss_D i loss_G.

**Jakość:** po ~10–15 epokach cyfry/ubrania są rozpoznawalne (MNIST zwykle szybciej). Epoki 1–5: szum; 10–15: ostrzejsze kształty.

**Porównanie:** MNIST – stabilniejszy trening; Fashion-MNIST – trudniejsze tekstury.

**Ograniczenia:** możliwy mode collapse przy zbyt silnym D lub złym LR.


**Cel zadania 2**

1. Załaduj zbiór CIFAR-10:
   * przeskaluj wartości pikseli do zakresu ([0,1]),
   * podziel dane na zbiór treningowy, testowy i walidacyjny.

2. Wygeneruj syntetyczny zbiór danych przy użyciu funkcji `make_blobs` z scikit-learn:

   * liczba próbek: min. 5000,
   * liczba cech: 77 (im więcej tym lepiej),
   * liczba klastrów: 32.

3. Zwizualizuj dane syntetyczne:

   * zredukuj wymiarowość do 2D (np. PCA),
   * wykonaj wykres punktowy z kolorowaniem względem etykiet.

## Autoenkoder dla CIFAR-10
   * zawiera co najmniej 2 warstwy konwolucyjne w encoderze,
   * redukuje reprezentację do przestrzeni latentnej o wymiarze 64,
   * rekonstruuje obraz o oryginalnym rozmiarze.---

## Autoenkoder dla danych syntetycznych (autoenkoder MLP)
   * encoder redukujący dane z 10 wymiarów do 2,
   * decoder rekonstruujący dane do oryginalnego wymiaru.

# Ewaluacja

Dla CIFAR-10:
  * pokaż przykłady: oryginalny obraz vs rekonstrukcja,
  * oceń jakość rekonstrukcji wizualnie i numerycznie (MSE).

Dla danych syntetycznych:
  * porównaj wartości wejściowe i wyjściowe,
  * oblicz średni błąd rekonstrukcji.

## Przestrzeń latentna

Dla danych syntetycznych:
  * zwizualizuj przestrzeń latentną (2D scatter plot),
  * pokoloruj punkty według rzeczywistych klastrów,
  * oceń, czy struktura została zachowana.

Dla CIFAR-10:
  * zredukuj latent space do 2D (PCA lub t-SNE),
  * oceń separację klas.


Zmień wymiar przestrzeni latentnej:
  * dla danych syntetycznych: 2 → 5,
  * dla CIFAR: 64 → 128,
  * porównaj wpływ na rekonstrukcję i strukturę latentną.

Dodaj regularyzację (np. dropout lub L2):
  * sprawdź wpływ na overfitting,
  * porównaj błędy walidacyjne.

---


